In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt').read().splitlines()

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [4]:
block_size = 3

def build_dataset(words):

    X, Y = [], []

    for w in words:

        context = [0] * block_size

        for ch in w + '.':
            ix = stoi[ch]

            X.append(context)
            Y.append(ix)

            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

In [5]:
import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [6]:
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [8]:
n_embd = 10
n_hidden = 200
vocab_size = len(stoi)

g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)

W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd*block_size)**0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1

W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

bngain = torch.randn((1, n_hidden)) * 0.1 + 1
bnbias = torch.randn((1, n_hidden)) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

In [9]:
batch_size = 32
n = batch_size

In [10]:
ix = torch.randint(0, Xtr.shape[0], (batch_size,))
Xb, Yb = Xtr[ix], Ytr[ix]

# embedding
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)

# layer 1
hprebn = embcat @ W1 + b1

# batchnorm
bnmean = hprebn.mean(0, keepdim=True)
bnvar = hprebn.var(0, keepdim=True, unbiased=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = (hprebn - bnmean) * bnvar_inv
hpreact = bngain * bnraw + bnbias

# nonlinearity
h = torch.tanh(hpreact)

# output layer
logits = h @ W2 + b2

loss = F.cross_entropy(logits, Yb)
loss

tensor(3.4194, grad_fn=<NllLossBackward0>)

In [11]:
dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n

In [12]:
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

In [13]:
dhpreact = (1 - h**2) * dh

In [14]:
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnbias = dhpreact.sum(0, keepdim=True)

dhprebn = bngain * bnvar_inv / n * (
    n * dhpreact
    - dhpreact.sum(0)
    - bnraw * (dhpreact * bnraw).sum(0)
)

In [15]:
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

In [16]:
demb = dembcat.view(emb.shape)

dC = torch.zeros_like(C)

for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix_ = Xb[k, j]
        dC[ix_] += demb[k, j]

In [19]:
def cmp(s, dt, t):
    tgrad = t.grad

    if tgrad is None:
        print(f"{s:15s} | t.grad is None (manual mode)")
        return

    ex = torch.all(dt == tgrad).item()
    app = torch.allclose(dt, tgrad)
    maxdiff = (dt - tgrad).abs().max().item()

    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [20]:
cmp("logits", dlogits, logits)
cmp("W2", dW2, W2)
cmp("b2", db2, b2)
cmp("W1", dW1, W1)
cmp("b1", db1, b1)
cmp("C", dC, C)

logits          | t.grad is None (manual mode)
W2              | t.grad is None (manual mode)
b2              | t.grad is None (manual mode)
W1              | t.grad is None (manual mode)
b1              | t.grad is None (manual mode)
C               | t.grad is None (manual mode)


/var/folders/lg/6_ftpksn6lgg9jbrnbmyz1hr0000gn/T/ipykernel_13320/3319293508.py:2: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  tgrad = t.grad


In [21]:
lr = 0.1

with torch.no_grad():
    C += -lr * dC
    W1 += -lr * dW1
    b1 += -lr * db1
    W2 += -lr * dW2
    b2 += -lr * db2
    bngain += -lr * dbngain
    bnbias += -lr * dbnbias

In [22]:
max_steps = 200000
batch_size = 32
n = batch_size

for i in range(max_steps):

    # --------------------
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # -------------------- forward
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)

    hprebn = embcat @ W1 + b1

    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5)**-0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias

    h = torch.tanh(hpreact)

    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # -------------------- backward
    dlogits = F.softmax(logits, 1)
    dlogits[range(n), Yb] -= 1
    dlogits /= n

    dh = dlogits @ W2.T
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)

    dhpreact = (1 - h**2) * dh

    dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)

    dhprebn = bngain * bnvar_inv / n * (
        n * dhpreact
        - dhpreact.sum(0)
        - bnraw * (dhpreact * bnraw).sum(0)
    )

    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)

    demb = dembcat.view(emb.shape)

    dC = torch.zeros_like(C)

    for k in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
            ix_ = Xb[k, j]
            dC[ix_] += demb[k, j]

    # -------------------- update
    lr = 0.1 if i < 100000 else 0.01

    with torch.no_grad():
        C += -lr * dC
        W1 += -lr * dW1
        b1 += -lr * db1
        W2 += -lr * dW2
        b2 += -lr * db2
        bngain += -lr * dbngain
        bnbias += -lr * dbnbias

    # -------------------- log
    if i % 10000 == 0:
        print(i, loss.item())

0 3.7183306217193604
10000 2.29581880569458
20000 2.025398015975952
30000 2.563833475112915
40000 2.1155803203582764
50000 2.4751157760620117
60000 2.1911277770996094
70000 2.1312460899353027
80000 2.037895917892456
90000 2.1892952919006348
100000 1.8315894603729248
110000 2.1898205280303955
120000 1.7468234300613403
130000 2.1455211639404297
140000 2.306601047515869
150000 1.8689392805099487
160000 2.1747148036956787
170000 2.143320322036743
180000 2.513554096221924
190000 2.1156206130981445


In [23]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]

    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)

    hprebn = embcat @ W1 + b1
    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)

    hpreact = bngain * (hprebn - bnmean) / torch.sqrt(bnvar + 1e-5) + bnbias
    h = torch.tanh(hpreact)

    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)

    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0709333419799805
val 2.111835241317749


In [24]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    out = []
    context = [0] * block_size

    while True:
        emb = C[torch.tensor([context])]
        embcat = emb.view(1, -1)

        hpreact = embcat @ W1 + b1
        hpreact = bngain * (hpreact - bnmean) / torch.sqrt(bnvar + 1e-5) + bnbias

        h = torch.tanh(hpreact)
        logits = h @ W2 + b2

        probs = F.softmax(logits, 1)
        ix = torch.multinomial(probs, 1).item()

        context = context[1:] + [ix]
        out.append(ix)

        if ix == 0:
            break

    print(''.join(itos[i] for i in out))

ter.
ijahailah.
zey.
giorna.
jayce.
dhareen.
fay.
ann.
ann.
maiclyn.
renn.
meghn.
utd.
jose.
karthorman.
luh.
ceryn.
jackxis.
jera.
kaiah.
